# 🧠🤖 Tutorial #1 — Train Your Own Model (Interactive)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atifhalim/BrainWerks/blob/main/learning/notebooks/Tutorial1_Train_Model.ipynb)

The sequel to the *Explore the Data* notebook. Here you actually **train an AI model**:

1. **Pick a task** (a dataset with an answer to learn).
2. **Pick a model** (an AI architecture).
3. **Set the training settings** (learning rate, passes, batch size).
4. Click **🚀 Train** and watch it learn — then see the **test accuracy**.

### How to run it
1. **Runtime ▸ Run all**.
2. *(Optional but faster:)* **Runtime ▸ Change runtime type ▸ GPU**, then Run all again.
3. Scroll to the bottom, choose your options, and press **🚀 Train**.

> First time you pick a *real* task it downloads a small sample (a few seconds).

## Step 1 — Set up (run once)

In [ ]:
!pip -q install mne braindecode ipywidgets

## Step 2 — The tasks (datasets with answers to learn)

A model can only learn if each clip has a **label** (the right answer, its *class*).
Each task below turns brain signals into `X` (the clips) and `y` (the labels).

In [ ]:
import numpy as np, mne, warnings
warnings.filterwarnings("ignore"); mne.set_log_level("ERROR")

WIN_S = 2.0          # seconds per clip
_data_cache = {}

def _windows(raw, picks, win_s=WIN_S):
    raw.pick(picks)
    sf = raw.info["sfreq"]; n = int(win_s * sf)
    d = raw.get_data() * 1e6
    segs = d.shape[1] // n
    return np.stack([d[:, i*n:(i+1)*n] for i in range(segs)]).astype("float32"), sf

def load_synthetic():
    rng = np.random.default_rng(0)
    X = (rng.standard_normal((80, 3, 320)) * 8).astype("float32")
    y = rng.integers(0, 4, 80).astype("int64")
    return X, y, ["class 0", "class 1", "class 2", "class 3"], \
        "Random noise with random labels — there is NO pattern, so accuracy should stay near chance (25%)."

def load_alpha():
    Xs, ys = [], []
    for run, lab in [(1, 0), (2, 1)]:      # 1 = eyes open, 2 = eyes closed
        fn = mne.datasets.eegbci.load_data(1, [run], update_path=True)
        raw = mne.io.read_raw_edf(fn[0], preload=True); mne.datasets.eegbci.standardize(raw)
        X, _ = _windows(raw, ["O1", "Oz", "O2", "P3", "Pz", "P4"])
        Xs.append(X); ys += [lab] * len(X)
    return np.concatenate(Xs), np.array(ys, "int64"), ["eyes open", "eyes closed"], \
        "Real EEG: eyes open vs eyes closed. Alpha waves surge when eyes close, so this is usually easy to learn."

def load_motor():
    raws = []
    for run in [4, 8, 12]:                 # imagined left / right fist
        fn = mne.datasets.eegbci.load_data(1, [run], update_path=True)
        r = mne.io.read_raw_edf(fn[0], preload=True); mne.datasets.eegbci.standardize(r); raws.append(r)
    raw = mne.concatenate_raws(raws); raw.pick(["C3", "Cz", "C4"])
    events, eid = mne.events_from_annotations(raw)
    ep = mne.Epochs(raw, events, {k: eid[k] for k in ("T1", "T2")},
                    tmin=0.0, tmax=2.0 - 1/raw.info["sfreq"], baseline=None, preload=True)
    X = (ep.get_data() * 1e6).astype("float32")
    y = (ep.events[:, -1] == eid["T2"]).astype("int64")   # 0 = left, 1 = right
    return X, y, ["left hand", "right hand"], \
        "Real EEG: a person imagined moving their LEFT vs RIGHT hand. Subtle and noisy — hard but real!"

TASKS = {
    "Synthetic (random noise, 4 classes)": load_synthetic,
    "Real: eyes open vs closed (alpha)":   load_alpha,
    "Real: imagine LEFT vs RIGHT hand":    load_motor,
}

def get_task(name):
    if name not in _data_cache:
        _data_cache[name] = TASKS[name]()
    return _data_cache[name]

print("Tasks ready:", *TASKS, sep="\n  • ")

## Step 3 — The models and the training routine

Three real braindecode models to choose from. The training routine splits the data,
trains, and scores on a **held-out test set** the model never saw.

In [ ]:
import torch
from braindecode.models import ShallowFBCSPNet, Deep4Net, EEGNet
from braindecode import EEGClassifier
from braindecode.util import set_random_seeds
from skorch.dataset import ValidSplit
from sklearn.model_selection import train_test_split

MODELS = {"ShallowFBCSPNet (recommended)": "Shallow",
          "Deep4Net (bigger)":            "Deep",
          "EEGNet (compact)":             "EEGNet"}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Training will run on:", DEVICE.upper(), "(tip: Runtime ▸ Change runtime type ▸ GPU for speed)")

def build_and_train(task_name, model_label, lr, epochs, batch_size, log=print):
    set_random_seeds(20240205, cuda=(DEVICE == "cuda"))
    X, y, classes, note = get_task(task_name)
    n_cls, C, T = len(classes), X.shape[1], X.shape[2]
    log(f"Task: {task_name}")
    log(note)
    log(f"data X = {X.shape}  (clips, channels, time) · {n_cls} classes · chance = {100/n_cls:.0f}%\n")

    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)
    key = MODELS[model_label]
    Model = {"Shallow": ShallowFBCSPNet, "Deep": Deep4Net, "EEGNet": EEGNet}[key]
    mod = (Model(n_chans=C, n_outputs=n_cls, n_times=T, final_conv_length="auto")
           if key in ("Shallow", "Deep") else Model(n_chans=C, n_outputs=n_cls, n_times=T))

    clf = EEGClassifier(mod, criterion=torch.nn.CrossEntropyLoss,
                        optimizer=torch.optim.Adam, optimizer__lr=lr,
                        batch_size=batch_size, max_epochs=epochs,
                        train_split=ValidSplit(0.2), device=DEVICE, verbose=1)
    log(f"Training {model_label} for {epochs} passes (lr={lr:g}, batch={batch_size})…\n")
    clf.fit(Xtr, ytr)
    test_acc = clf.score(Xte, yte)
    return clf, test_acc, n_cls, classes, (len(Xtr), len(Xte))

print("Models:", *MODELS, sep="\n  • ")

## Step 4 — Train it! 👇

**train / validation / test** — three separate piles of clips:
* **Training** clips: the model *learns* from these.
* **Validation** clips: checked *during* training (the `valid_acc` column) to watch progress.
* **Test** clips: locked away until the very end, then used **once** for the honest score.

*(Reminder: here, an "epoch" = one full pass through the training clips — not an EEG epoch.)*

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
try:
    from google.colab import output as _o; _o.enable_custom_widget_manager()
except Exception:
    pass

w_task  = widgets.Dropdown(options=list(TASKS),  description="Task:",  layout=widgets.Layout(width="480px"), style={"description_width":"initial"})
w_model = widgets.Dropdown(options=list(MODELS), description="Model:", layout=widgets.Layout(width="480px"), style={"description_width":"initial"})
w_lr    = widgets.FloatLogSlider(value=6.25e-4, base=10, min=-4, max=-2, step=0.1, description="Learning rate:", readout_format=".4f", style={"description_width":"initial"}, layout=widgets.Layout(width="480px"))
w_ep    = widgets.IntSlider(value=25, min=1, max=100, step=1, description="Training passes (epochs):", style={"description_width":"initial"}, layout=widgets.Layout(width="480px"))
w_bs    = widgets.Dropdown(options=[8, 16, 32, 64], value=16, description="Batch size:", style={"description_width":"initial"})
btn     = widgets.Button(description="🚀 Train", button_style="success")
out     = widgets.Output()

def _run(_):
    with out:
        clear_output()
        print("⏳ Training… (first real task downloads a small sample)\n")
        try:
            clf, acc, n_cls, classes, (ntr, nte) = build_and_train(
                w_task.value, w_model.value, w_lr.value, w_ep.value, w_bs.value)
            chance = 1.0 / n_cls
            print(f"\n──────── RESULT ────────")
            print(f"trained on {ntr} clips · tested on {nte} unseen clips")
            print(f"TEST ACCURACY = {acc*100:.1f}%   (chance = {chance*100:.0f}%)")
            if acc >= chance + 0.15:
                print("🎉 Well above chance — the model really learned the pattern!")
            elif acc >= chance + 0.05:
                print("🙂 A bit above chance — it learned something. Try more passes or another model.")
            else:
                print("😐 Around chance — like the random-noise task, it found no real pattern here.")
            fig, ax = plt.subplots(figsize=(5, 2.6))
            ax.bar(["chance", "this model"], [chance*100, acc*100], color=["#9aa0b5", "#22a06b"])
            ax.set_ylabel("accuracy (%)"); ax.set_ylim(0, 100)
            for i, v in enumerate([chance*100, acc*100]):
                ax.text(i, v+2, f"{v:.0f}%", ha="center")
            plt.tight_layout(); plt.show()
        except Exception as e:
            print("⚠️ Something went wrong:", type(e).__name__, str(e)[:300])

btn.on_click(_run)
display(widgets.VBox([w_task, w_model, w_lr, w_ep, w_bs, btn, out]))

## Try these experiments 🔬

* **Start with "eyes open vs closed"** and ShallowFBCSPNet, ~25 passes — you should get well above 50%.
* **Now try the "random noise" task** — no matter what you do, it stays near 25%. That's the Tutorial #1 lesson: no pattern, no learning.
* **Try "imagine LEFT vs RIGHT hand"** — much harder! Bump the passes up, try a different model, see how high you can get.
* **Change the learning rate** — too high and it won't settle; too low and it barely learns. There's a sweet spot.
* **Change the model** — does Deep4Net or EEGNet beat Shallow on your task?

Every run splits the data fresh, trains, and scores on clips the model never saw — the honest way to measure learning.